# Workshop Part 1: Data Extraction (ETL)

## Learning Objectives
- Understand the Extract phase of ETL pipelines
- Load data from CSV files (simulating extraction from external sources)
- Perform initial data exploration
- Identify data quality issues

## Scenario
Albert Heijn wants to analyze customer reviews of fresh products to determine which products need discounts based on customer sentiment. First, we need to extract and examine the review data.

In [32]:
# Workshop Progress Tracker
notebooks = ["01 Extract", "02 Prepare", "03 Storage", "04 ML", "05 Deploy"]
current = 0  # This is notebook 01

print("="*70)
print("📊 WORKSHOP PROGRESS")
print("="*70)
for i, nb in enumerate(notebooks):
    if i < current:
        print(f"✅ {nb}")
    elif i == current:
        print(f"👉 {nb} ← YOU ARE HERE")
    else:
        print(f"⬜ {nb}")
print("="*70)

📊 WORKSHOP PROGRESS
👉 01 Extract ← YOU ARE HERE
⬜ 02 Prepare
⬜ 03 Storage
⬜ 04 ML
⬜ 05 Deploy


## Step 1: Import Required Libraries

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully!")

Libraries imported successfully!


## Step 2: Extract Data from Multiple Sources

### The Real-World Scenario

In a real company like Albert Heijn, data comes from **different systems**:
- **📝 Customer Reviews** - From the feedback/rating system
- **📦 Inventory Levels** - From the warehouse management system
- **💰 Sales Data** - From point-of-sale (POS) terminals

**Important:** In real ETL pipelines, data is rarely in one convenient table!

In this workshop, we simulate these sources with three separate CSV files.

In [34]:
# Load the three separate datasets
print("="*70)
print("📊 EXTRACTING DATA FROM MULTIPLE SOURCES")
print("="*70)
print("\nLoading data from three different systems...")

# 1. Customer Reviews (from feedback system)
reviews = pd.read_csv('../data/raw/reviews.csv')
print(f"\n📝 REVIEWS: {len(reviews)} customer reviews loaded")
print(f"   Columns: {list(reviews.columns)}")

# 2. Inventory Levels (from warehouse system)
inventory = pd.read_csv('../data/raw/inventory.csv')
print(f"\n📦 INVENTORY: {len(inventory)} daily inventory records loaded")
print(f"   Columns: {list(inventory.columns)}")

# 3. Sales Data (from POS terminals)
sales = pd.read_csv('../data/raw/sales.csv')
print(f"\n💰 SALES: {len(sales)} daily sales records loaded")
print(f"   Columns: {list(sales.columns)}")

print("\n" + "="*70)
print("✅ All data sources extracted successfully!")
print("="*70)

📊 EXTRACTING DATA FROM MULTIPLE SOURCES

Loading data from three different systems...

📝 REVIEWS: 773 customer reviews loaded
   Columns: ['product_id', 'product_name', 'date', 'rating', 'review_text']

📦 INVENTORY: 414 daily inventory records loaded
   Columns: ['product_id', 'product_name', 'date', 'stock_level']

💰 SALES: 407 daily sales records loaded
   Columns: ['product_id', 'product_name', 'date', 'sales_volume']

✅ All data sources extracted successfully!


## Step 3: Explore Each Dataset

Let's examine each dataset separately before merging them.

In [35]:
# Preview Reviews dataset
print("📝 REVIEWS Dataset Preview:")
print("="*70)
print(reviews.head(5))
print(f"\nShape: {reviews.shape[0]} rows × {reviews.shape[1]} columns")

📝 REVIEWS Dataset Preview:
   product_id product_name        date  rating  \
0           4     Tomatoes  2025-10-28       4   
1           1     Potatoes  2025-10-15       3   
2           6     Broccoli  2025-12-10       4   
3           6     Broccoli  2025-12-01       3   
4           2      Carrots  2025-11-09       3   

                                         review_text  
0    Love these tomatoes! Always consistent quality.  
1   Average quality potatoes. Not bad but not great.  
2      Superb quality! These broccoli are wonderful.  
3  The broccoli are alright, nothing to complain ...  
4             The carrots are okay, I've had better.  

Shape: 773 rows × 5 columns


In [36]:
# Preview Inventory dataset
print("📦 INVENTORY Dataset Preview:")
print("="*70)
print(inventory.head(5))
print(f"\nShape: {inventory.shape[0]} rows × {inventory.shape[1]} columns")

📦 INVENTORY Dataset Preview:
   product_id product_name        date  stock_level
0           1     Potatoes  2025-10-09           65
1           1     Potatoes  2025-10-10           89
2           1     Potatoes  2025-10-12          100
3           1     Potatoes  2025-10-14          137
4           1     Potatoes  2025-10-15          156

Shape: 414 rows × 4 columns


In [37]:
# Preview Sales dataset
print("💰 SALES Dataset Preview:")
print("="*70)
print(sales.head(5))
print(f"\nShape: {sales.shape[0]} rows × {sales.shape[1]} columns")

💰 SALES Dataset Preview:
   product_id product_name        date  sales_volume
0           1     Potatoes  2025-10-09           134
1           1     Potatoes  2025-10-10           368
2           1     Potatoes  2025-10-12           350
3           1     Potatoes  2025-10-14           202
4           1     Potatoes  2025-10-15           234

Shape: 407 rows × 4 columns


## Step 4: Data Integration Challenge 🎯

### The Problem

We have three separate datasets, but we need **one combined view** to analyze:
- Which products with bad reviews also have low sales?
- Do we have enough stock for products getting positive reviews?
- Should we discount products with low ratings and excess inventory?

**Solution:** We need to MERGE the datasets using common keys!

### 🧩 Merge Step 1: Combine Reviews + Inventory

Let's merge reviews with inventory to see the stock level on the day each review was written.

In [38]:
# Merge reviews with inventory on product_id, product_name, and date
print("🔄 Merging: Reviews + Inventory...")
print("="*70)

reviews_inventory = reviews.merge(
    inventory,
    on=['product_id', 'product_name', 'date'],
    how='left'  # Keep all reviews, even if no matching inventory data
)

print(f"✅ Merge complete!")
print(f"   Input:  {len(reviews)} reviews + {len(inventory)} inventory records")
print(f"   Result: {len(reviews_inventory)} rows")
print(f"\n💡 KEY INSIGHT: We used 'product_id + product_name + date' as merge keys")
print(f"   This matches each review with the stock level on that specific day.")

print("\nSample of merged data:")
print(reviews_inventory[['product_name', 'date', 'rating', 'stock_level']].head(5))

🔄 Merging: Reviews + Inventory...
✅ Merge complete!
   Input:  773 reviews + 414 inventory records
   Result: 773 rows

💡 KEY INSIGHT: We used 'product_id + product_name + date' as merge keys
   This matches each review with the stock level on that specific day.

Sample of merged data:
  product_name        date  rating  stock_level
0     Tomatoes  2025-10-28       4          127
1     Potatoes  2025-10-15       3          156
2     Broccoli  2025-12-10       4          108
3     Broccoli  2025-12-01       3          133
4      Carrots  2025-11-09       3          159


### 🧩 Merge Step 2: Add Sales Data

Now let's add sales data to complete the picture!

In [39]:
# Merge with sales data
print("🔄 Merging: (Reviews + Inventory) + Sales...")
print("="*70)

df = reviews_inventory.merge(
    sales,
    on=['product_id', 'product_name', 'date'],
    how='left'  # Keep all reviews
)

print(f"✅ Final merge complete!")
print(f"   Input:  {len(reviews_inventory)} rows + {len(sales)} sales records")
print(f"   Result: {len(df)} rows")
print(f"\n🎉 SUCCESS: We now have a complete dataset with reviews, inventory, AND sales!")

print("\n📊 Complete dataset preview:")
print(df[['product_name', 'date', 'rating', 'review_text', 'stock_level', 'sales_volume']].head(5))

🔄 Merging: (Reviews + Inventory) + Sales...
✅ Final merge complete!
   Input:  773 rows + 407 sales records
   Result: 773 rows

🎉 SUCCESS: We now have a complete dataset with reviews, inventory, AND sales!

📊 Complete dataset preview:
  product_name        date  rating  \
0     Tomatoes  2025-10-28       4   
1     Potatoes  2025-10-15       3   
2     Broccoli  2025-12-10       4   
3     Broccoli  2025-12-01       3   
4      Carrots  2025-11-09       3   

                                         review_text  stock_level  \
0    Love these tomatoes! Always consistent quality.          127   
1   Average quality potatoes. Not bad but not great.          156   
2      Superb quality! These broccoli are wonderful.          108   
3  The broccoli are alright, nothing to complain ...          133   
4             The carrots are okay, I've had better.          159   

   sales_volume  
0         213.0  
1         234.0  
2         313.0  
3         233.0  
4         119.0  


### 📊 Understanding the Complete Dataset

Now that we've merged all three sources, let's understand what we have:

In [40]:
# Get basic information about the merged dataset
print("Dataset Info:")
print("=" * 70)
df.info()

print("\n" + "=" * 70)
print("Column Descriptions:")
print("=" * 70)
print("  • product_id:    Unique identifier for each product")
print("  • product_name:  Name of the fresh product")
print("  • date:          Date of the review")
print("  • rating:        Star rating (1-5)")
print("  • review_text:   Customer review text")
print("  • stock_level:   Inventory on that date (from warehouse)")
print("  • sales_volume:  Units sold on that date (from POS)")

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 773 entries, 0 to 772
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    773 non-null    int64  
 1   product_name  773 non-null    object 
 2   date          773 non-null    object 
 3   rating        773 non-null    int64  
 4   review_text   773 non-null    object 
 5   stock_level   773 non-null    int64  
 6   sales_volume  766 non-null    float64
dtypes: float64(1), int64(3), object(3)
memory usage: 42.4+ KB

Column Descriptions:
  • product_id:    Unique identifier for each product
  • product_name:  Name of the fresh product
  • date:          Date of the review
  • rating:        Star rating (1-5)
  • review_text:   Customer review text
  • stock_level:   Inventory on that date (from warehouse)
  • sales_volume:  Units sold on that date (from POS)


In [41]:
# Statistical summary
print("Statistical Summary:")
print("=" * 70)
df.describe()

Statistical Summary:


,product_id,rating,stock_level,sales_volume
count,773.000000,773.000000,773.000000,766.000000
mean,3.673997,3.712807,173.129366,288.946475
std,1.694951,1.214252,49.148462,89.989954
min,1.000000,1.000000,51.000000,62.000000
25%,2.000000,3.000000,137.000000,233.250000
50%,4.000000,4.000000,172.000000,284.000000
75%,5.000000,5.000000,208.000000,348.000000
max,6.000000,5.000000,300.000000,567.000000


## Step 5: Business Insights from Merged Data 💡

Now that we have all data combined, let's answer real business questions!

In [42]:
# Example business question with merged data
print("="*70)
print("🎯 BUSINESS INSIGHTS FROM INTEGRATED DATA")
print("="*70)

# Question 1: Find an interesting example
spinach_data = df[(df['product_name'] == 'Spinach') & df['stock_level'].notna() & df['sales_volume'].notna()]

if len(spinach_data) > 0:
    example = spinach_data.iloc[0]
    
    print(f"\n📊 Example: {example['product_name']} on {example['date']}")
    print(f"   ⭐ Rating: {example['rating']}/5 stars")
    print(f"   📝 Review: {example['review_text'][:60]}...")
    print(f"   📦 Stock: {int(example['stock_level'])} units in warehouse")
    print(f"   💰 Sales: {int(example['sales_volume'])} units sold that day")
    
    # Calculate stock coverage
    days_of_stock = example['stock_level'] / example['sales_volume']
    print(f"   📈 Stock Coverage: {days_of_stock:.1f} days")
    
    if days_of_stock < 2:
        print(f"   ⚠️  WARNING: Low stock! Risk of stockout!")
    elif days_of_stock > 7:
        print(f"   ⚠️  WARNING: Overstock! Risk of food waste!")
    else:
        print(f"   ✅ Good stock level!")

print("\n" + "="*70)
print("💡 THIS IS THE POWER OF DATA INTEGRATION!")
print("   Without merging, we couldn't answer: 'Should we discount based")
print("   on reviews, stock, AND sales together?'")
print("="*70)

🎯 BUSINESS INSIGHTS FROM INTEGRATED DATA

📊 Example: Spinach on 2025-12-05
   ⭐ Rating: 4/5 stars
   📝 Review: Love these spinach! Always consistent quality....
   📦 Stock: 201 units in warehouse
   💰 Sales: 410 units sold that day
   📈 Stock Coverage: 0.5 days
   ⚠️  WARNING: Low stock! Risk of stockout!

💡 THIS IS THE POWER OF DATA INTEGRATION!
   Without merging, we couldn't answer: 'Should we discount based
   on reviews, stock, AND sales together?'


### 🔍 BECOME A DATA DETECTIVE!

Instead of just showing you the results, let's explore the data together!

In [43]:
# Make exploration active
print("="*70)
print("🔍 BECOME A DATA DETECTIVE!")
print("="*70)
print("\n📋 YOUR MISSION: Explore the dataset and answer these questions")
print("   Use pandas functions like .value_counts(), .isnull().sum(), .describe()")
print("\n❓ QUESTIONS:")
print("   1. Which product has the MOST reviews?")
print("   2. How many reviews are missing text? (Critical for sentiment analysis!)")
print("   3. What's the most common rating (1-5 stars)?")
print("   4. Are there any duplicate reviews? How many?")
print("\n💡 TIP: Each question needs 1-2 lines of code")
print("   Example: df['column_name'].value_counts()")
print("="*70)
print("\n👇 Write your detective code in the cells below!")

🔍 BECOME A DATA DETECTIVE!

📋 YOUR MISSION: Explore the dataset and answer these questions
   Use pandas functions like .value_counts(), .isnull().sum(), .describe()

❓ QUESTIONS:
   1. Which product has the MOST reviews?
   2. How many reviews are missing text? (Critical for sentiment analysis!)
   3. What's the most common rating (1-5 stars)?
   4. Are there any duplicate reviews? How many?

💡 TIP: Each question needs 1-2 lines of code
   Example: df['column_name'].value_counts()

👇 Write your detective code in the cells below!


In [44]:
# Question 1: Which product has the MOST reviews?
# Your code here:


In [45]:
# Question 2: How many reviews are missing text?
# Your code here:


In [46]:
# Question 3: What's the most common rating?
# Your code here:


In [47]:
# Question 4: Are there any duplicate reviews?
# Your code here:


**⏱️ Take 3 minutes:** Work individually or with your neighbor to answer the questions above.

### 🎯 Detective Answers

Run the cell below to check your answers!

In [48]:
# ANSWERS (run after students try)
print("="*70)
print("🎯 DETECTIVE ANSWERS")
print("="*70)
print(f"\n1. Most reviews: {df['product_name'].value_counts().index[0]}")
print(f"   ({df['product_name'].value_counts().iloc[0]} reviews)")
print(f"\n2. Missing text: {df['review_text'].isnull().sum()} reviews")
print(f"   ({(df['review_text'].isnull().sum() / len(df) * 100):.1f}% of data)")
print(f"\n3. Most common rating: {df['rating'].mode()[0]}⭐")
print(f"   (appears {(df['rating'] == df['rating'].mode()[0]).sum()} times)")
print(f"\n4. Duplicates: {df.duplicated().sum()} duplicate rows found")
print(f"   ({(df.duplicated().sum() / len(df) * 100):.1f}% of data)")
print("\n" + "="*70)
print("✅ How did you do? Compare with your neighbor!")
print("💬 Discuss: Why are missing reviews and duplicates a problem?")

🎯 DETECTIVE ANSWERS

1. Most reviews: Broccoli
   (150 reviews)

2. Missing text: 0 reviews
   (0.0% of data)

3. Most common rating: 5⭐
   (appears 261 times)

4. Duplicates: 33 duplicate rows found
   (4.3% of data)

✅ How did you do? Compare with your neighbor!
💬 Discuss: Why are missing reviews and duplicates a problem?


## 🎯 Quick Recap: Test Your Understanding

Take 1-2 minutes to discuss these questions with your neighbor:

In [49]:
print("="*70)
print("🎯 QUICK RECAP: Test Your Understanding")
print("="*70)
print("\n❓ DISCUSSION QUESTIONS (Talk with your neighbor for 1-2 minutes):")
print("\n1. Why is it important to check data quality BEFORE analysis?")
print("   Hint: What happens if we train an ML model on bad data?")
print("\n2. What's worse: missing data or duplicate data? Why?")
print("   Hint: Think about their different impacts on analysis.")
print("\n3. Why can't we do sentiment analysis without review_text?")
print("   Hint: What does our ML model need as input?")
print("\n✅ Ready? Let's move to Notebook 02: Data Preparation!")
print("="*70)

🎯 QUICK RECAP: Test Your Understanding

❓ DISCUSSION QUESTIONS (Talk with your neighbor for 1-2 minutes):

1. Why is it important to check data quality BEFORE analysis?
   Hint: What happens if we train an ML model on bad data?

2. What's worse: missing data or duplicate data? Why?
   Hint: Think about their different impacts on analysis.

3. Why can't we do sentiment analysis without review_text?
   Hint: What does our ML model need as input?

✅ Ready? Let's move to Notebook 02: Data Preparation!


## Summary: What We Learned

### ✅ ETL - Extract Phase
We extracted data from **three separate sources** (simulating real systems):
1. Customer reviews from feedback system
2. Inventory levels from warehouse system
3. Sales data from POS terminals

### ✅ Data Integration
We **merged** datasets using common keys (`product_id`, `product_name`, `date`):
- Used `pandas.merge()` with `how='left'` to keep all reviews
- Combined three sources into one complete view

### ✅ Data Quality
We identified issues that need fixing:
- Missing review text (can't do sentiment analysis without it!)
- Missing sales/stock data (some dates don't match perfectly)
- Duplicate reviews

### 🎯 Business Value
With integrated data, we can now answer:
- "Should we discount products with bad reviews AND excess stock?"
- "Do we have enough inventory for products getting positive reviews?"
- "Which products need attention based on reviews, sales, AND stock?"

## Next Steps

In **Notebook 02 (Data Preparation)**, we will:
- Clean the merged data (handle missing values)
- Remove duplicate reviews
- Prepare text for machine learning
- Create sentiment labels
- Save the cleaned dataset

## Key Takeaways

- **ETL (Extract)**: The first step in any data pipeline is extracting data from sources
- **Data Exploration**: Always explore your data before analysis
- **Quality Checks**: Identify issues early (missing values, duplicates, inconsistencies)
- **Documentation**: Understanding your data structure is crucial for downstream tasks